In [1]:
pip install transformers torch datasets rouge-score pyarrow==21.0.0

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ------------------------ --------------- 524.3/846.4 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 846.4/846.4 kB 2.2 MB/s  0:00:00

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 2.2.0

    Uninstalling huggingface_hub-2.2.0:

   ---------------------------------------- 0/2 [huggingface-hub]
      Successfully uninstalled huggingface_hub-2.2.0
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------

In [ ]:

from transformers import pipeline
from datasets import load_dataset
from rouge_score import rouge_scorer

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

r1 = 0
r2 = 0
rl = 0

for i, item in enumerate(dataset):
    article = item["article"]
    reference = item["highlights"]

    summary = summarizer(
        article[:3000],
        max_length=100,
        min_length=30,
        do_sample=False,
        truncation=True
    )[0]["summary_text"]

    scores = scorer.score(reference, summary)

    r1 += scores["rouge1"].fmeasure
    r2 += scores["rouge2"].fmeasure
    rl += scores["rougeL"].fmeasure

    print("\nArticle", i + 1)
    print("Generated Summary:", summary)
    print("Reference Summary:", reference)
    print("ROUGE-1:", round(scores["rouge1"].fmeasure, 4))
    print("ROUGE-2:", round(scores["rouge2"].fmeasure, 4))
    print("ROUGE-L:", round(scores["rougeL"].fmeasure, 4))

n = len(dataset)

print("\nAverage ROUGE Scores")
print("ROUGE-1:", round(r1 / n, 4))
print("ROUGE-2:", round(r2 / n, 4))
print("ROUGE-L:", round(rl / n, 4))
